# Whisper Fine-Tuning for Dysarthric Speech

Fine-tunes `openai/whisper-small` on your labeled WAV files so it learns the speaker's pronunciation patterns.

**Data format:** Place WAV files in `data/raw/` where the filename (without `.wav`) is exactly what the speaker says.
Example: `The sky is blue.wav`

**Re-running:** Just add more WAV files to `data/raw/` and run all cells again — the notebook auto-discovers everything.

**Output:** Fine-tuned model saved to `models/whisper_finetuned/`, available as `whisper_finetuned` in `evaluate.py`.

## Step 1: Install Dependencies

Run once. Skip if already installed.

In [1]:
!pip install -q transformers datasets peft evaluate jiwer audiomentations torchaudio soundfile accelerate librosa


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Step 2: Configuration

Adjust these settings as your dataset grows:
- Increase `MAX_TRAIN_STEPS` when you have more data (rule of thumb: ~50 steps per clip)
- `AUGMENTATION_FACTOR` expands each clip N times with noise/speed/pitch variations

In [2]:
from pathlib import Path

# --- Paths ---
DATA_DIR    = "data/raw"                # WAV files; filename stem = transcript
OUTPUT_DIR  = "models/whisper_finetuned"  # where to save the fine-tuned model

# --- Model ---
BASE_MODEL  = "openai/whisper-small"    # options: whisper-tiny, whisper-base, whisper-small, whisper-medium

# --- Training ---
MAX_TRAIN_STEPS            = 10   # increase with more data: ~50 steps per clip is a good starting point
LEARNING_RATE              = 1e-4
BATCH_SIZE                 = 2
GRADIENT_ACCUMULATION      = 4     # effective batch = BATCH_SIZE * GRADIENT_ACCUMULATION
WARMUP_STEPS               = 50

# --- Data augmentation ---
AUGMENTATION_FACTOR        = 8     # augmented copies per original clip (helps with tiny datasets)

# --- LoRA (recommended for small datasets, prevents overfitting) ---
USE_LORA    = True
LORA_RANK   = 16
LORA_ALPHA  = 32

print(f"Data directory : {Path(DATA_DIR).resolve()}")
print(f"Output directory: {Path(OUTPUT_DIR).resolve()}")
print(f"Base model      : {BASE_MODEL}")

Data directory : /Users/arjanverma/development/claude/cp_speech/data/raw
Output directory: /Users/arjanverma/development/claude/cp_speech/models/whisper_finetuned
Base model      : openai/whisper-small


## Step 3: Load Dataset

Discovers all WAV files in `data/raw/` and derives the ground truth from each filename.

In [3]:
import re

data_dir  = Path(DATA_DIR)
wav_files = sorted(data_dir.glob("*.wav"))

if not wav_files:
    raise FileNotFoundError(
        f"No .wav files found in '{DATA_DIR}'.\n"
        "Add files named like 'hello world.wav' where the filename = what the speaker says."
    )

def stem_to_label(stem: str) -> str:
    """Remove trailing .-N suffixes: 'The sky is blue.-0' -> 'The sky is blue'."""
    return re.sub(r'\.\-\d+$', '', stem).strip()

samples = [
    {"audio_path": str(p), "sentence": stem_to_label(p.stem)}
    for p in wav_files
]

print(f"Found {len(samples)} labeled clips:\n")
for s in samples:
    size_kb = Path(s['audio_path']).stat().st_size // 1024
    print(f"  [{s['sentence']}]")
    print(f"    {Path(s['audio_path']).name}  ({size_kb} KB)\n")

Found 6 labeled clips:

  [Effects of the first world war]
    Effects of the first world war.wav  (636 KB)

  [For producing the things]
    For producing the things.wav  (480 KB)

  [The prices doubled]
    The prices doubled.wav  (345 KB)

  [The sky is blue]
    The sky is blue.-0.wav  (144 KB)

  [in England were setup by seventeenthirty]
    in England were setup by seventeenthirty.wav  (784 KB)

  [that the sour and bitter taste]
    that the sour and bitter taste.wav  (581 KB)



## Step 4: Data Augmentation

Expands the small dataset by creating multiple variations of each clip with:
- **Gaussian noise** — simulates different recording environments
- **Time stretch** — slight speed variations (±15%)
- **Pitch shift** — slight pitch variations (±2 semitones)

The original clips are kept in a separate eval set (no augmentation — we measure real performance).

In [4]:
import numpy as np
import librosa
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift

SAMPLE_RATE = 16000  # Whisper requires 16kHz

def load_audio(path: str) -> np.ndarray:
    """Load audio file resampled to 16kHz mono."""
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    return audio

augment = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.5),
    TimeStretch(min_rate=0.85, max_rate=1.15, p=0.5),
    PitchShift(min_semitones=-2, max_semitones=2, p=0.5),
])

print("Loading audio files...")
train_audio = []  # augmented clips for training
eval_audio  = []  # original clips for evaluation

for s in samples:
    audio = load_audio(s["audio_path"])
    label = s["sentence"]

    # Original goes to eval set only
    eval_audio.append({"audio": audio, "sentence": label})

    # Create AUGMENTATION_FACTOR augmented copies for training
    for _ in range(AUGMENTATION_FACTOR):
        aug_audio = augment(audio.copy(), sample_rate=SAMPLE_RATE)
        train_audio.append({"audio": aug_audio, "sentence": label})

print(f"Training set : {len(train_audio)} augmented clips ({AUGMENTATION_FACTOR}x per original)")
print(f"Eval set     : {len(eval_audio)} original clips")

Loading audio files...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training set : 48 augmented clips (8x per original)
Eval set     : 6 original clips


## Step 5: Extract Whisper Features

Converts raw audio waveforms to log-mel spectrograms and tokenizes text labels — the format Whisper expects.

In [5]:
from transformers import WhisperProcessor
from datasets import Dataset

print(f"Loading Whisper processor: {BASE_MODEL}")
processor = WhisperProcessor.from_pretrained(BASE_MODEL, language="English", task="transcribe")

def extract_features(audio: np.ndarray, label: str) -> dict:
    """Extract mel features and tokenize label for one clip."""
    # Log-mel spectrogram: shape (80, 3000) — fixed size for Whisper (30s window)
    feats = processor.feature_extractor(
        audio, sampling_rate=SAMPLE_RATE, return_tensors="np"
    ).input_features[0]
    tokens = processor.tokenizer(label).input_ids
    return {"input_features": feats, "labels": tokens}

print("Extracting features for training set...")
train_data = [extract_features(s["audio"], s["sentence"]) for s in train_audio]

print("Extracting features for eval set...")
eval_data  = [extract_features(s["audio"], s["sentence"]) for s in eval_audio]

train_dataset = Dataset.from_list(train_data)
eval_dataset  = Dataset.from_list(eval_data)

print(f"\nTrain dataset: {len(train_dataset)} samples")
print(f"Eval dataset : {len(eval_dataset)} samples")
print(f"Feature shape: {np.array(train_data[0]['input_features']).shape}  (mel_bins x time_frames)")

Loading Whisper processor: openai/whisper-small


Extracting features for training set...
Extracting features for eval set...

Train dataset: 48 samples
Eval dataset : 6 samples
Feature shape: (80, 3000)  (mel_bins x time_frames)


## Step 6: Load Model + Apply LoRA

LoRA (Low-Rank Adaptation) adds small trainable matrices to the attention layers without changing the base model weights.
This prevents overfitting on small datasets — only ~1.2M parameters are trained instead of 244M.

The script auto-detects the best available device: Apple Silicon MPS > CUDA GPU > CPU.

In [6]:
import torch
from transformers import WhisperForConditionalGeneration

# --- Device selection ---
if torch.backends.mps.is_available():
    device = torch.device("mps")
    device_name = "Apple Silicon MPS"
elif torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = torch.cuda.get_device_name(0)
else:
    device = torch.device("cpu")
    device_name = "CPU (slow — reduce MAX_TRAIN_STEPS if needed)"

print(f"Device: {device_name}")
print(f"\nLoading {BASE_MODEL}...")

model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)

# Required config for fine-tuning
model.config.use_cache = False  # must be False when using gradient checkpointing
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="english", task="transcribe"
)
model.config.suppress_tokens = []

if USE_LORA:
    from peft import LoraConfig, get_peft_model
    model.enable_input_require_grads()
    lora_cfg = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        target_modules=["q_proj", "v_proj"],  # attention projection layers
        lora_dropout=0.05,
        bias="none",
    )
    model = get_peft_model(model, lora_cfg)
    trainable, total = model.get_nb_trainable_parameters()
    print(f"\nLoRA applied:")
    print(f"  Trainable params : {trainable:>12,}")
    print(f"  Total params     : {total:>12,}")
    print(f"  Trainable %      : {100*trainable/total:.2f}%")
else:
    total = sum(p.numel() for p in model.parameters())
    print(f"\nFull fine-tune: {total:,} params (consider USE_LORA=True for small datasets)")

Device: Apple Silicon MPS

Loading openai/whisper-small...


Loading weights: 100%|██████████| 479/479 [00:00<00:00, 1637.19it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]  



LoRA applied:
  Trainable params :    1,769,472
  Total params     :  243,504,384
  Trainable %      : 0.73%


## Step 7: Training Setup

Sets up the data collator, WER metric, and HuggingFace `Seq2SeqTrainer`.

In [7]:
from dataclasses import dataclass
from typing import Any, Dict, List
import sys
import importlib
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# The project has a local evaluate.py that shadows the HuggingFace 'evaluate' package.
# Load the installed package directly by temporarily removing '.' from sys.path.
_saved_path = sys.path[:]
sys.path = [p for p in sys.path if p not in ("", ".")]
hf_evaluate = importlib.import_module("evaluate")
sys.path = _saved_path

# --- Data collator ---
# Whisper input features are always (80, 3000) — no padding needed.
# Only labels need padding (different transcript lengths).
@dataclass
class WhisperDataCollator:
    processor: Any

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        # Stack fixed-size mel features
        input_features = torch.tensor(
            np.stack([f["input_features"] for f in features]),
            dtype=torch.float32
        )
        # Pad label sequences with -100 (ignored by CrossEntropyLoss)
        label_seqs = [torch.tensor(f["labels"], dtype=torch.long) for f in features]
        labels = torch.nn.utils.rnn.pad_sequence(
            label_seqs, batch_first=True, padding_value=-100
        )
        return {"input_features": input_features, "labels": labels}

data_collator = WhisperDataCollator(processor=processor)

# --- Metric: Word Error Rate ---
wer_metric = hf_evaluate.load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids.copy()
    # Replace -100 padding back to pad_token_id before decoding
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": round(wer, 4)}

# --- Training arguments ---
use_fp16 = torch.cuda.is_available()  # fp16 on CUDA only; MPS uses fp32 for stability

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=MAX_TRAIN_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    fp16=use_fp16,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=225,
    report_to="none",
    dataloader_num_workers=0,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

eff_batch = BATCH_SIZE * GRADIENT_ACCUMULATION
print(f"Effective batch size : {eff_batch}")
print(f"Training steps       : {MAX_TRAIN_STEPS}")
print(f"Mixed precision fp16 : {use_fp16}")
print("Trainer ready.")


Effective batch size : 8
Training steps       : 10
Mixed precision fp16 : False
Trainer ready.


## Step 8: Train

This may take a few minutes on Apple Silicon. Watch the WER column — it should decrease.
The best checkpoint (lowest WER) is loaded automatically at the end.

In [8]:
print(f"Starting fine-tuning — {MAX_TRAIN_STEPS} steps...")
train_result = trainer.train()

print("\nTraining complete!")
print(f"  Final training loss : {train_result.training_loss:.4f}")
print(f"  Total steps         : {train_result.global_step}")

Starting fine-tuning — 10 steps...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss



Training complete!
  Final training loss : 29.2401
  Total steps         : 10


## Step 9: Evaluate — Baseline vs Fine-Tuned

Runs both the original `whisper-small` and your fine-tuned model on the **original, unaugmented** clips
and prints a side-by-side WER comparison.

In [9]:
import pandas as pd
from jiwer import wer as compute_wer
from transformers import pipeline as hf_pipeline

# --- Baseline: original whisper-small ---
print(f"Loading baseline model ({BASE_MODEL}) for comparison...")
baseline_pipe = hf_pipeline(
    "automatic-speech-recognition",
    model=BASE_MODEL,
    torch_dtype=torch.float32,
    device="cpu",  # keep on CPU so we don't conflict with the fine-tuned model
)

# --- Fine-tuned model inference ---
model.eval()
model.to(device)

rows = []
for s in eval_audio:
    label = s["sentence"]
    audio = s["audio"]

    # Baseline prediction
    baseline_text = baseline_pipe(
        {"array": audio, "sampling_rate": SAMPLE_RATE}
    )["text"].strip()

    # Fine-tuned prediction
    feats = processor.feature_extractor(
        audio, sampling_rate=SAMPLE_RATE, return_tensors="pt"
    ).input_features.to(device)
    with torch.no_grad():
        gen_ids = model.generate(feats)
    finetuned_text = processor.tokenizer.decode(gen_ids[0], skip_special_tokens=True).strip()

    b_wer = compute_wer(label.lower(), baseline_text.lower())
    f_wer = compute_wer(label.lower(), finetuned_text.lower())

    rows.append({
        "Ground Truth"   : label,
        "Baseline"       : baseline_text,
        "B-WER"          : round(b_wer, 3),
        "Fine-Tuned"     : finetuned_text,
        "FT-WER"         : round(f_wer, 3),
    })

df = pd.DataFrame(rows)

# Display
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.width", 140)
print("\n" + df.to_string(index=False))

avg_b = df["B-WER"].mean()
avg_f = df["FT-WER"].mean()
delta = avg_b - avg_f

print(f"\n{'─'*60}")
print(f"  Avg Baseline WER   : {avg_b:.3f}")
print(f"  Avg Fine-Tuned WER : {avg_f:.3f}")
improvement_pct = 100 * delta / avg_b if avg_b > 0 else 0
print(f"  Improvement        : {delta:+.3f} WER pts ({improvement_pct:+.1f}%)")
print(f"{'─'*60}")

Loading baseline model (openai/whisper-small) for comparison...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 479/479 [00:00<00:00, 1376.99it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]  
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsPro

ValueError: You have modified the pretrained model configuration to control generation We detected the following values set - {'suppress_tokens': []}. This strategy to control generation is not supported anymore. Please use and modify `model.generation_config` (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )

## Step 10: Save Model

Merges the LoRA weights back into the base model and saves a self-contained model to `models/whisper_finetuned/`.
This can be loaded directly by the `whisper_finetuned` provider in `evaluate.py`.

In [ ]:
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

if USE_LORA and hasattr(model, "merge_and_unload"):
    print("Merging LoRA weights into base model...")
    merged_model = model.merge_and_unload()
    merged_model.save_pretrained(OUTPUT_DIR)
    print(f"Merged model saved to: {OUTPUT_DIR}/")
else:
    trainer.save_model(OUTPUT_DIR)
    print(f"Model saved to: {OUTPUT_DIR}/")

processor.save_pretrained(OUTPUT_DIR)
print(f"Processor saved to: {OUTPUT_DIR}/")

# Summary of saved files
saved_files = list(Path(OUTPUT_DIR).glob("*"))
print(f"\nSaved {len(saved_files)} files:")
for f in sorted(saved_files):
    size_mb = f.stat().st_size / (1024*1024)
    print(f"  {f.name:40s}  {size_mb:.1f} MB")

print(f"""
{'='*55}
Fine-tuning complete!

Next steps:
  1. Add more WAV files to data/raw/ and re-run to improve
  2. Evaluate all providers:
       python evaluate.py
  3. The 'whisper_finetuned' provider is now available
{'='*55}
""")